# Standalone Inference Notebook — PM2.5 Forecasting

**Purpose:** Load a pre-trained `best_model.pt` (from a previous Kaggle run) and generate a correct `preds.npy` submission file.

**How to use on Kaggle:**
1. Add **three** datasets to this notebook:
   - Competition data (`aisehack-theme-2`) — contains `test_in/`, `raw/`, `stats/`
   - Source dataset (`murtuza-pm25-src30`) — contains `Murtuza/src/`, `Murtuza/configs/`
   - A dataset containing your saved `best_model.pt` — upload it separately
2. Update `BEST_MODEL_PATH` in Cell 1 to point to where `best_model.pt` is mounted
3. Run all cells — `preds.npy` will appear in `/kaggle/working/`

**Key preprocessing fixes vs the original `pino_baseline.ipynb`:**
- December grid-wise sigma normalization is **NOT applied** to test cpm25  
  (training used `SlidingWindowTensorDataset` which always uses standard min-max log1p)
- Unknown future cpm25 hours (11-16) are filled via **persistence** (repeat last known value)  
  instead of zeros, better matching the training distribution

In [ ]:
# ── Cell 1: Bootstrap — paths, imports ─────────────────────────────────────
import os, sys
import numpy as np
import torch

# ── Kaggle paths — adjust MURTUZA_DIR and BEST_MODEL_PATH as needed ─────────
KAGGLE_DATA_ROOT  = "/kaggle/input/datasets/khushisingh942004/aisehack"
MURTUZA_DIR       = "/kaggle/input/datasets/sasidhar412/murtuza-pm25-src30/ANRF_AISEHack_Code/Murtuza"

# ⚠️  Update this to the mount path of your uploaded best_model.pt dataset:
BEST_MODEL_PATH   = "/kaggle/input/murtuza-best-model/best_model.pt"

OUT_DIR           = "/kaggle/working"
PRED_PATH         = os.path.join(OUT_DIR, "preds.npy")

# ── Local fallback (non-Kaggle debugging) ────────────────────────────────────
if not os.path.exists("/kaggle"):
    MURTUZA_DIR     = os.path.abspath("../Murtuza")
    KAGGLE_DATA_ROOT = os.path.abspath("../../aisehack-theme-2")
    BEST_MODEL_PATH = os.path.abspath("../outputs/models/best_model.pt")
    OUT_DIR         = os.path.abspath("../outputs/submissions")
    PRED_PATH       = os.path.join(OUT_DIR, "preds.npy")

if MURTUZA_DIR not in sys.path:
    sys.path.insert(0, MURTUZA_DIR)

os.makedirs(OUT_DIR, exist_ok=True)

assert os.path.exists(MURTUZA_DIR),     f"Source dir not found: {MURTUZA_DIR}"
assert os.path.exists(KAGGLE_DATA_ROOT), f"Data root not found: {KAGGLE_DATA_ROOT}"
assert os.path.exists(BEST_MODEL_PATH), f"best_model.pt not found: {BEST_MODEL_PATH}\n→ Upload your best_model.pt as a Kaggle dataset and update BEST_MODEL_PATH above."

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"Data   : {KAGGLE_DATA_ROOT}")
print(f"Source : {MURTUZA_DIR}")
print(f"Model  : {BEST_MODEL_PATH}")
print(f"Output : {PRED_PATH}")

In [ ]:
# ── Cell 2: Load config + official bounds ───────────────────────────────────
from src.config import load_config
from src.data import load_minmax_bounds, FALLBACK_OFFICIAL_BOUNDS

cfg = load_config()
cfg['paths']['data'] = KAGGLE_DATA_ROOT
cfg['device'] = DEVICE

bounds = load_minmax_bounds(cfg)
fmin_cpm = bounds['cpm25'][0]
fmax_cpm = bounds['cpm25'][1]

print(f"cpm25 bounds: [{fmin_cpm}, {fmax_cpm}]")
print(f"Current config base features: {cfg['features']['base']}")
print(f"Config loaded. Months: {cfg['data']['months']}")

In [ ]:
# ── Cell 3: Inspect checkpoint, infer channel layout, build model ────────────
from src.model import build_model

state = torch.load(BEST_MODEL_PATH, map_location=DEVICE)

# Infer expected flat input channels directly from checkpoint.
if 'lift.weight' in state:
    expected_flat = int(state['lift.weight'].shape[1])
else:
    expected_flat = int(cfg.get('tensor_channels', 26 * cfg['time'].get('t_in', 16)))

T_MODEL = int(cfg['time'].get('t_in', 16))
if expected_flat % T_MODEL != 0:
    raise RuntimeError(f"Checkpoint input channels {expected_flat} not divisible by T_MODEL={T_MODEL}")

N_CHANNELS = expected_flat // T_MODEL

# Legacy checkpoint support:
#   current preprocessing: 15 base + 11 engineered = 26
#   legacy preprocessing : 16 base + 12 engineered = 28
LEGACY_BASE_FEATS = [
    'cpm25', 'u10', 'v10', 'pblh', 'rain', 't2', 'q2', 'swdown', 'psfc',
    'PM25', 'SO2', 'NOx', 'NH3', 'NMVOC_e', 'NMVOC_finn', 'bio',
]
CURRENT_BASE_FEATS = list(cfg['features']['base'])

if N_CHANNELS == 26:
    BASE_FEATS = CURRENT_BASE_FEATS
    ENGINEER_MODE = 'hotspot26'
elif N_CHANNELS == 28:
    BASE_FEATS = LEGACY_BASE_FEATS
    ENGINEER_MODE = 'legacy28'
    for feat in BASE_FEATS:
        if feat not in bounds:
            bounds[feat] = FALLBACK_OFFICIAL_BOUNDS[feat]
else:
    raise RuntimeError(
        f"Unsupported checkpoint channel count: {N_CHANNELS}. "
        "This notebook currently supports 26-channel current checkpoints and 28-channel legacy checkpoints."
    )

cfg['tensor_channels'] = expected_flat
model = build_model(cfg).to(DEVICE)
model.load_state_dict(state)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Checkpoint flat in-channels : {expected_flat}")
print(f"T_MODEL                    : {T_MODEL}")
print(f"N_CHANNELS                 : {N_CHANNELS}")
print(f"Preprocessing mode         : {ENGINEER_MODE}")
print(f"Base feature count         : {len(BASE_FEATS)}")
print(f"Model type                 : {cfg['model']['type']}")
print(f"Parameters                 : {total_params:,}")
print(f"Weights loaded             : {BEST_MODEL_PATH}")

In [ ]:
# ── Cell 4: Preprocess test data ────────────────────────────────────────────
import gc
from src.data import _transform_bounds, preprocess_feature

TEST_DIR = os.path.join(KAGGLE_DATA_ROOT, 'test_in')
N_TEST   = cfg['data']['test_samples']
T_IN_CPM = cfg['time']['t_in_cpm']
T_IN_MET = cfg['time']['t_in_met']
H, W     = 140, 124

base_arrays = {}

for feat in BASE_FEATS:
    raw = np.load(os.path.join(TEST_DIR, f'{feat}.npy'), mmap_mode='r').astype(np.float32)

    if feat == 'cpm25':
        cpm_log = np.log1p(np.clip(raw, 0.0, None))
        lo_t, hi_t = _transform_bounds(fmin_cpm, fmax_cpm, 'cpm25')
        cpm_norm = np.clip((cpm_log - lo_t) / (hi_t - lo_t + 1e-8), 0.0, 1.0).astype(np.float32)
        last = cpm_norm[:, -1:, :, :]
        fill = np.repeat(last, T_IN_MET - T_IN_CPM, axis=1)
        base_arrays['cpm25'] = np.concatenate([cpm_norm, fill], axis=1)
        print(f"  cpm25        {base_arrays['cpm25'].shape}  [min-max log1p + persistence fill]")
        del cpm_log, cpm_norm, last, fill
        gc.collect()
        continue

    base_arrays[feat] = preprocess_feature(raw, feat, bounds, month=None)
    print(f"  {feat:<12} {base_arrays[feat].shape}")
    gc.collect()

x_full = np.stack([base_arrays[f] for f in BASE_FEATS], axis=1).astype(np.float32)
print(f"\nFull test tensor: {x_full.shape}  ({x_full.nbytes / 1e9:.2f} GB)")

x_test = x_full[:, :, :T_MODEL, :, :]
del x_full, base_arrays
gc.collect()
print(f"Test tensor sliced to T_MODEL={T_MODEL}: {x_test.shape}")

In [ ]:
# ── Cell 5: Append engineered channels to match checkpoint layout ───────────
from src.data import get_hotspot_maps

N, C, T, Hg, Wg = x_test.shape
feat_idx = {name: i for i, name in enumerate(BASE_FEATS)}

u10   = x_test[:, feat_idx['u10']]
v10   = x_test[:, feat_idx['v10']]
t2    = x_test[:, feat_idx['t2']]
q2    = x_test[:, feat_idx['q2']]
cpm25 = x_test[:, feat_idx['cpm25']]

wind_speed = np.sqrt(u10 ** 2 + v10 ** 2)
wind_dir   = np.arctan2(v10, u10)

denom_safe = np.where(
    np.abs(t2 - np.float32(29.65)) < np.float32(1e-3),
    np.sign(t2 - np.float32(29.65) + np.float32(1e-3)) * np.float32(1e-3),
    t2 - np.float32(29.65),
).astype(np.float32)
exponent = np.clip(
    np.float32(17.67) * (t2 - np.float32(273.15)) / denom_safe,
    np.float32(-87.0), np.float32(87.0),
)
with np.errstate(over='ignore', invalid='ignore'):
    rh = q2 / (np.float32(0.622) + np.float32(0.378) * q2 + np.float32(1e-8)) * np.exp(exponent)
np.nan_to_num(rh, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
rh = np.clip(rh, np.float32(0.0), np.float32(1.5))

hour      = (np.arange(T, dtype=np.float32) % 24)
hour_sin  = np.broadcast_to(np.sin(np.float32(2*np.pi) * hour / 24.0).reshape(1, T, 1, 1), (N, T, Hg, Wg)).copy()
hour_cos  = np.broadcast_to(np.cos(np.float32(2*np.pi) * hour / 24.0).reshape(1, T, 1, 1), (N, T, Hg, Wg)).copy()
month_sin = np.full((N, T, Hg, Wg), np.sin(np.float32(2*np.pi / 12.0)), dtype=np.float32)
month_cos = np.full((N, T, Hg, Wg), np.cos(np.float32(2*np.pi / 12.0)), dtype=np.float32)

lag_1 = np.roll(cpm25, 1, axis=1)
lag_3 = np.roll(cpm25, 3, axis=1)
lag_6 = np.roll(cpm25, 6, axis=1)

if ENGINEER_MODE == 'hotspot26':
    hotspot_prior, _ = get_hotspot_maps(cfg)
    hotspot = np.broadcast_to(hotspot_prior.reshape(1, 1, Hg, Wg), (N, T, Hg, Wg)).copy().astype(np.float32)
    engineered = np.stack([
        wind_speed, wind_dir, rh,
        hour_sin, hour_cos, month_sin, month_cos,
        hotspot, lag_1, lag_3, lag_6,
    ], axis=1).astype(np.float32)
elif ENGINEER_MODE == 'legacy28':
    ll = np.load(os.path.join(KAGGLE_DATA_ROOT, 'raw', 'lat_long.npy')).astype(np.float32)
    lat = ll[:, :, 0]
    lon = ll[:, :, 1]
    lat = (lat - lat.min()) / (lat.max() - lat.min() + np.float32(1e-8))
    lon = (lon - lon.min()) / (lon.max() - lon.min() + np.float32(1e-8))
    lat = np.broadcast_to(lat.reshape(1, 1, Hg, Wg), (N, T, Hg, Wg)).copy().astype(np.float32)
    lon = np.broadcast_to(lon.reshape(1, 1, Hg, Wg), (N, T, Hg, Wg)).copy().astype(np.float32)
    engineered = np.stack([
        wind_speed, wind_dir, rh,
        hour_sin, hour_cos, month_sin, month_cos,
        lat, lon, lag_1, lag_3, lag_6,
    ], axis=1).astype(np.float32)
else:
    raise RuntimeError(f"Unsupported engineer mode: {ENGINEER_MODE}")

x_test = np.concatenate([x_test, engineered], axis=1).astype(np.float32)
np.nan_to_num(x_test, copy=False, nan=0.0, posinf=0.0, neginf=0.0)

assert x_test.shape == (N_TEST, N_CHANNELS, T_MODEL, H, W), (
    f"Expected ({N_TEST}, {N_CHANNELS}, {T_MODEL}, {H}, {W}), got {x_test.shape}"
)
print(f"Final test tensor: {x_test.shape}")
print(f"  Mode           : {ENGINEER_MODE}")
print(f"  NaN/Inf count  : {int(np.sum(~np.isfinite(x_test)))}")
print(f"  Value range    : [{x_test.min():.3f}, {x_test.max():.3f}]")

In [ ]:
# ── Cell 6: Batched inference ────────────────────────────────────────────────
from src.data import denormalize

BATCH_SIZE = cfg['training']['batch_size_test']  # 16
all_preds  = []

model.eval()
with torch.no_grad():
    for i in range(0, N_TEST, BATCH_SIZE):
        j     = min(i + BATCH_SIZE, N_TEST)
        batch = torch.from_numpy(x_test[i:j]).to(DEVICE)  # (B, 26, 16, 140, 124)

        pred_norm = model(batch)                                         # (B, 140, 124, 16)  normalized
        pred_phys = denormalize(pred_norm.cpu().numpy(), fmin_cpm, fmax_cpm, feat='cpm25')
        pred_phys = np.clip(pred_phys, 0.0, None)                        # PM2.5 >= 0
        all_preds.append(pred_phys)

        if i == 0:
            print(f"[Batch 0] input {batch.shape}  → output {pred_norm.shape}")
            print(f"          pred range: [{pred_phys.min():.1f}, {pred_phys.max():.1f}] µg/m³")
        if (i // BATCH_SIZE) % 5 == 0:
            print(f"  Processed {j}/{N_TEST} samples ...", end="\r")

preds = np.concatenate(all_preds, axis=0).astype(np.float32)
print(f"\nInference complete.")
print(f"Output shape : {preds.shape}")
print(f"Value range  : [{preds.min():.1f}, {preds.max():.1f}] µg/m³")
print(f"Mean PM2.5   : {preds.mean():.2f} µg/m³")

assert preds.shape == (N_TEST, H, W, T_MODEL), f"Shape mismatch: {preds.shape}"
assert np.isfinite(preds).all(), "Predictions contain NaN/Inf — check preprocessing!"

In [ ]:
# ── Cell 7: Verify, save, and confirm preds.npy ─────────────────────────────
import time

tmp_path = PRED_PATH + '.tmp'

# Atomic write: tmp → fsync → rename
with open(tmp_path, 'wb') as f:
    np.save(f, preds)
    f.flush()
    os.fsync(f.fileno())
os.replace(tmp_path, PRED_PATH)

# Read-back verification
loaded = np.load(PRED_PATH, mmap_mode='r')
assert loaded.shape == preds.shape, f"Read-back shape mismatch: {loaded.shape}"
assert np.allclose(loaded[:4], preds[:4], atol=1e-5), "Read-back values differ!"

size_mb = os.path.getsize(PRED_PATH) / (1024**2)
print("═" * 55)
print(f"  preds.npy saved to : {PRED_PATH}")
print(f"  Shape              : {loaded.shape}")
print(f"  dtype              : {loaded.dtype}")
print(f"  File size          : {size_mb:.2f} MB")
print(f"  Value range        : [{preds.min():.1f}, {preds.max():.1f}] µg/m³")
print(f"  Mean PM2.5         : {preds.mean():.2f} µg/m³")
print(f"  All finite         : {np.isfinite(preds).all()}")
print("═" * 55)
print("Ready to submit. Download preds.npy from /kaggle/working/ and upload to the competition.")